## Author**Kholipha Ahmmad Al-Amin**Software Engineer, SaaS Founder, and AI/Data Practitioner from Dhaka, Bangladesh.Portfolio: https://kholipha-ahmmad-al-amin.equisaas-bd.comGitHub: https://github.com/kholipha-ahmmad-al-aminLinkedIn: https://www.linkedin.com/in/kholipha-ahmmad-al-amin

<a href="https://colab.research.google.com/github/anms5519/AI-Image-Enhancer-Advanced/blob/main/Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install PySide6 onnxruntime pillow numpy opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.5/550.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.6/160.6 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.3/95.3 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.2 MB/s eta 0:00:00


In [ ]:
# main.py

import sys
import os
from PySide6.QtWidgets import (
    QApplication,
    QMainWindow,
    QLabel,
    QPushButton,
    QFileDialog,
    QVBoxLayout,
    QWidget,
    QMessageBox,
)
from PySide6.QtGui import QPixmap
from PySide6.QtCore import Qt, QPropertyAnimation, QRect
import onnxruntime as ort
import numpy as np
from PIL import Image, ImageEnhance, ImageQt

###########################################
# Deep Learning Model Integration Section #
###########################################

def load_model(model_path="image_enhancer.onnx"):
    """
    Load the ONNX model using ONNX Runtime.
    Raises FileNotFoundError if the model file is not found.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    ort_session = ort.InferenceSession(model_path)
    return ort_session

def preprocess_image(image, input_size=(224, 224)):
    """
    Preprocess the image:
      - Resize to the model's expected input size.
      - Convert to numpy array with values in [0, 1].
      - Rearrange dimensions to (batch, channels, height, width).
    """
    image_resized = image.resize(input_size)
    img_array = np.array(image_resized).astype(np.float32) / 255.0  # Normalize to [0,1]
    img_array = np.transpose(img_array, (2, 0, 1))  # Change to CHW
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    return img_array

def postprocess_image(output_array):
    """
    Convert the model output (assumed to be in shape (1, C, H, W) and normalized)
    back to a PIL Image.
    """
    output_array = output_array.squeeze(0)  # Remove batch dimension
    output_array = np.transpose(output_array, (1, 2, 0))  # Change to HWC
    output_array = np.clip(output_array * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(output_array)

def run_inference(ort_session, image):
    """
    Run the ONNX model inference on the provided image.
    """
    input_array = preprocess_image(image, input_size=(224, 224))
    input_name = ort_session.get_inputs()[0].name
    outputs = ort_session.run(None, {input_name: input_array})
    enhanced_image = postprocess_image(outputs[0])
    return enhanced_image

def dummy_enhance(image):
    """
    A fallback function to simulate image enhancement.
    (Increases sharpness and brightness.)
    """
    enhancer = ImageEnhance.Sharpness(image)
    image = enhancer.enhance(2.0)
    enhancer = ImageEnhance.Brightness(image)
    image = enhancer.enhance(1.2)
    return image

###########################################
# User Interface (UI) Code Section        #
###########################################

class ImageEnhancerUI(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Legendary Image Enhancer")
        self.setGeometry(100, 100, 800, 600)

        # Main widget and layout
        self.central_widget = QWidget()
        self.layout = QVBoxLayout()

        # Label for displaying images
        self.image_label = QLabel("Load an image to enhance")
        self.image_label.setAlignment(Qt.AlignCenter)
        self.layout.addWidget(self.image_label)

        # Button to load an image
        self.load_button = QPushButton("Load Image")
        self.load_button.clicked.connect(self.load_image)
        self.layout.addWidget(self.load_button)

        # Button to enhance the image
        self.enhance_button = QPushButton("Enhance Image")
        self.enhance_button.clicked.connect(self.enhance_image)
        self.layout.addWidget(self.enhance_button)

        self.central_widget.setLayout(self.layout)
        self.setCentralWidget(self.central_widget)

        # Attempt to load the ONNX model
        self.model_loaded = False
        try:
            self.ort_session = load_model("image_enhancer.onnx")
            self.model_loaded = True
            print("Deep learning model loaded successfully!")
        except Exception as e:
            print("ONNX model not loaded. Falling back to dummy enhancement. Error:", e)
            self.ort_session = None

        self.loaded_image = None  # Placeholder for the loaded PIL image

    def load_image(self):
        """
        Opens a file dialog to load an image and displays it.
        """
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Select Image", "", "Image Files (*.png *.jpg *.jpeg)"
        )
        if file_path:
            try:
                self.loaded_image = Image.open(file_path).convert("RGB")
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Failed to load image:\n{e}")
                return

            qt_image = ImageQt.ImageQt(self.loaded_image)
            pixmap = QPixmap.fromImage(qt_image)
            self.image_label.setPixmap(pixmap.scaled(600, 400, Qt.KeepAspectRatio))

    def enhance_image(self):
        """
        Enhances the loaded image using the deep learning model or dummy function,
        then displays the enhanced image with a simple animation.
        """
        if self.loaded_image is None:
            QMessageBox.warning(self, "No Image", "Please load an image first!")
            return

        # Create a quick animation for visual feedback
        anim = QPropertyAnimation(self.image_label, b"geometry")
        anim.setDuration(500)
        anim.setStartValue(self.image_label.geometry())
        anim.setEndValue(
            QRect(
                self.image_label.x(),
                self.image_label.y(),
                self.image_label.width() - 50,
                self.image_label.height() - 50,
            )
        )
        anim.setLoopCount(2)
        anim.start()
        QApplication.processEvents()  # Update UI during the animation

        # Run the deep learning inference if available, otherwise use dummy enhancement
        try:
            if self.model_loaded and self.ort_session is not None:
                enhanced_image = run_inference(self.ort_session, self.loaded_image)
            else:
                enhanced_image = dummy_enhance(self.loaded_image)
        except Exception as e:
            QMessageBox.warning(self, "Enhancement Error", f"Error during enhancement:\n{e}")
            return

        # Update the UI with the enhanced image
        qt_image = ImageQt.ImageQt(enhanced_image)
        pixmap = QPixmap.fromImage(qt_image)
        self.image_label.setPixmap(pixmap.scaled(600, 400, Qt.KeepAspectRatio))

###########################################
# Main Execution Section                  #
###########################################

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = ImageEnhancerUI()
    window.show()
    sys.exit(app.exec())
